# Imports

In [1]:
# !python3 -m venv venv
!source venv/bin/activate

In [2]:
# !pip install matplotlib

# !pip install scvi-tools
# !pip install scanpy
# !pip install torch torchvision torchaudio
# # if only CPU available
# # !pipx install jax
# # if Nvidia GPU available
# !pip install "jax[cuda12]"

# !pip install -U scarches
# !pip install pandas==2.3.1
# !pip install pilotpy
# !pip install scikit-misc
# !pip install psutil

# !pip install numpy==1.26.4

In [4]:
# Imports
import os
import time
import psutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scvi
import seaborn as sns
from scvi.external import MRVI
from scarches.models.scpoli import scPoli
import pilotpy as pl
from scipy.sparse import csr_matrix

import torch, jax; print(torch.cuda.is_available()); print(jax.devices())

scvi.settings.seed = 0  # optional: ensures reproducibility
print("Last run with scvi-tools version:", scvi.__version__)

True


Seed set to 0


[CudaDevice(id=0)]
Last run with scvi-tools version: 1.2.2.post2


In [5]:
def log_execution_time_and_memory(dataset_name, execution_time, dataset_memory, peak_memory, path_data, log_file_name='execution_times.txt'):
    log_file = os.path.join(path_data, log_file_name)

    if not os.path.exists(log_file):
        with open(log_file, 'w') as f:  # Open in write mode to create the file
            f.write("Dataset Processing Log\n")
            f.write("===========================\n")
    
    with open(log_file, 'a') as f:  # Open in append mode
        f.write(f"{dataset_name}: {execution_time} seconds, Dataset size: {dataset_memory} MB, Peak memory usage: {peak_memory} MB\n")

def process_data(dataset_name, ct_annotation, path_data = os.path.join(os.getcwd(), "data")):
    print(dataset_name)
    path_adata = os.path.join(path_data, dataset_name + ".h5ad")

    path_mrvi_dists = os.path.join(path_data, dataset_name + '_mrvi_dists.feather')
    path_scpoli_embs = os.path.join(path_data, dataset_name + '_scpoli_embs.feather')
    path_pilot_dists = os.path.join(path_data, dataset_name + '_pilot_dists.feather')

    file_paths_to_check = [
        path_mrvi_dists,
        path_scpoli_embs,
        path_pilot_dists
    ]

    if all(os.path.exists(path) for path in file_paths_to_check):
        return
    
    # Get initial memory usage before starting the task
    process = psutil.Process(os.getpid())
    initial_memory = process.memory_info().rss / (1024 * 1024)  # Convert bytes to MB
    
    adata = sc.read_h5ad(path_adata, backed='r+')
    adata = adata.to_memory() # Make sure the data is fully loaded into memory

    # Track peak memory usage after loading the data
    peak_memory = process.memory_info().rss / (1024 * 1024)  # Convert bytes to MB
    dataset_memory = peak_memory - initial_memory
    
    
    sc.pp.highly_variable_genes(adata, n_top_genes=2000, subset=True, flavor="seurat_v3")

    if not os.path.exists(path_mrvi_dists):
        print('Processing MrVI')
        start_time = time.time()
        
        adata.obs['dummy_col'] = np.zeros(adata.n_obs)
        MRVI.setup_anndata(adata, sample_key="Sample")
        model = MRVI(adata)
        model.train(max_epochs=50, accelerator="cpu")
        dists = model.get_local_sample_distances(keep_cell=False, groupby="dummy_col", batch_size=32)
        da = dists['dummy_col']
        df_dists = da.isel(dummy_col_name=0)
        df_dists = df_dists.to_pandas()
        df_dists.to_feather(path_mrvi_dists)
        
        end_time = time.time()
        execution_time = end_time - start_time
        
        # Update peak memory during processing
        peak_memory = max(peak_memory, process.memory_info().rss / (1024 * 1024)) - dataset_memory - initial_memory
        
        log_execution_time_and_memory(dataset_name, execution_time, dataset_memory, peak_memory, path_data)
    
    if not os.path.exists(path_scpoli_embs):
        print('Processing scPoli')
        start_time = time.time()
        
        # Required to speed up scpoli
        adata.X = adata.X.todense()
        adata.X = adata.X.astype('float32')
        early_stopping_kwargs = {
            "early_stopping_metric": "val_prototype_loss",
            "mode": "min",
            "threshold": 0,
            "patience": 20,
            "reduce_lr": True,
            "lr_patience": 13,
            "lr_factor": 0.1,
        }
        scpoli_model = scPoli(
            adata=adata,
            condition_keys='Sample',
            cell_type_keys=ct_annotation,
            embedding_dims=5,
            recon_loss='nb',
        )
        scpoli_model.train(
            n_epochs=50,
            pretraining_epochs=40,
            early_stopping_kwargs=early_stopping_kwargs,
            eta=5,
        )
        adata_emb = scpoli_model.get_conditional_embeddings()
        df_embs = pd.DataFrame(adata_emb.X, columns=[f"Dim_{i+1}" for i in range(adata_emb.X.shape[1])], index=adata_emb.obs_names)
        df_embs.to_feather(path_scpoli_embs)
        
        end_time = time.time()
        execution_time = end_time - start_time
        
        # Update peak memory during processing
        peak_memory = max(peak_memory, process.memory_info().rss / (1024 * 1024)) - dataset_memory - initial_memory
        
        log_execution_time_and_memory(dataset_name, execution_time, dataset_memory, peak_memory, path_data)

    if not os.path.exists(path_pilot_dists):
        print('Processing PILOT')
        adata = sc.read_h5ad(path_adata, backed='r+')
        adata = adata.to_memory() # Make sure the data is fully loaded into memory
        
        start_time = time.time()
        
        sc.pp.normalize_total(adata)
        sc.pp.log1p(adata)
        sc.pp.highly_variable_genes(adata, n_top_genes=2000, subset=True, flavor="seurat")
        sc.pp.scale(adata, max_value=10)
        sc.pp.pca(adata, n_comps=50)
        pl.tl.wasserstein_distance(
            adata,
            emb_matrix = 'X_pca',
            clusters_col = ct_annotation,
            sample_col = 'Sample',
            status = 'Sample',
        )
        df_dists = adata.uns['EMD_df']
        df_dists.to_feather(path_pilot_dists)
        
        end_time = time.time()
        execution_time = end_time - start_time
        
        # Update peak memory during processing
        peak_memory = max(peak_memory, process.memory_info().rss / (1024 * 1024)) - dataset_memory - initial_memory
        
        log_execution_time_and_memory(dataset_name, execution_time, dataset_memory, peak_memory, path_data)

In [6]:
datasets = {
    'Adams': 'OriginalAnnotationLevel1',
    'Bassez': 'cellType',
    'Gongsharma_cmv_females': 'AIFI_L1',
    'Gongsharma_age_females_cmvneg': 'AIFI_L1',
    'Kfoury': 'cells_lowres',
    'Lee': 'scGate_multi',
    'Pelka': 'OriginalAnnotationLevel1',
    'Smillie': 'OriginalAnnotationLevel2',
    'Stephenson': 'initial_clustering',
    'Wu': 'celltype_major',
    'Zhang': 'layer1',
}

for dataset_name, ct_annotation in datasets.items():
    process_data(dataset_name, ct_annotation)

Adams
Bassez
Gongsharma_cmv_females
Gongsharma_age_females_cmvneg
Kfoury
Processing MrVI
INFO     Jax module moved to TFRT_CPU_0.Note: Pytorch lightning will show GPU is not being used for the Trainer.   


GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Epoch 50/50: 100%|██████████████████| 50/50 [06:20<00:00,  7.25s/it, v_num=1, train_loss_step=397, train_loss_epoch=398]

`Trainer.fit` stopped: `max_epochs=50` reached.


Epoch 50/50: 100%|██████████████████| 50/50 [06:20<00:00,  7.60s/it, v_num=1, train_loss_step=397, train_loss_epoch=398]


...

Epoch 50/50: 100%|██████████████████| 50/50 [19:13<00:00, 23.07s/it, v_num=1, train_loss_step=400, train_loss_epoch=403]


100%|███████████████████████████████████████████████████████████████████████████████| 6259/6259 [11:52<00:00,  8.79it/s]


Processing scPoli


INFO:scarches.trainers.scpoli.trainer:GPU available: True, GPU used: True


Embedding dictionary:
 	Num conditions: [70]
 	Embedding dim: [5]
Encoder Architecture:
	Input Layer in, out and cond: 2000 45 5
	Mean/Var Layer in/out: 45 10
Decoder Architecture:
	First Layer in, out and cond:  10 45 5
	Output Layer in/out:  45 2000 

Initializing dataloaders
Starting training
 |████████████████████| 100.0%  - val_loss:  413.10 - val_cvae_loss:  407.03 - val_prototype_loss:    6.07 - val_labeled_loss:    1.21
Processing PILOT
Stephenson
Wu
Zhang
